# HAI Digital Twin — Full Pipeline

Runs every stage in order. Each cell is independent — you can re-run a single stage without re-running earlier ones.

**Environment:** Python 3.14 · torch 2.12 (CUDA 12.8) · numpy 2.2 · pandas 2.3 · scikit-learn 1.7 · dowhy 0.14

```
Step 0  Preprocess raw CSVs → scaled windows (.npz)          [skip if already done]
Step 1  Train stage 1: GRU-Causal+  (warm-start from Re_gru)
Step 2  Train stage 2: Scenario-weighted
Step 3  Evaluate both checkpoints (NRMSE tables)
Step 4  Attack detection  (ROC, PR, confusion matrix)
Step 5  Attack classification  (TRTS classifier → .pkl)
```

In [ ]:
import subprocess, sys, os
from pathlib import Path

ROOT = Path(".").resolve()
print("Working directory:", ROOT)
print("Python interpreter:", sys.executable)

import torch, numpy, pandas, sklearn, scipy, matplotlib, dowhy
print(f"torch      {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")
print(f"numpy      {numpy.__version__}")
print(f"pandas     {pandas.__version__}")
print(f"sklearn    {sklearn.__version__}")
print(f"scipy      {scipy.__version__}")
print(f"matplotlib {matplotlib.__version__}")
print(f"dowhy      {dowhy.__version__}")

def run(script, *args):
    """Run a project script, streaming stdout+stderr into the notebook cell."""
    cmd = [sys.executable, str(ROOT / script)] + list(args)
    print(f"\n{'='*60}")
    print(f"Running: {' '.join(cmd)}")
    print(f"{'='*60}\n")
    result = subprocess.run(
        cmd,
        cwd=ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(
            f"{script} exited with code {result.returncode} "
            f"(0x{result.returncode & 0xFFFFFFFF:08X})\n"
            "See output above for the actual error."
        )
    print(f"\n  {script} finished successfully.")

## Step 0 — Preprocess data
Reads raw HAI CSVs, fits the scaler on normal training data, and writes sliding-window `.npz` files to `outputs/scaled_split/`.

> **Skip this cell if `outputs/scaled_split/` already exists** — it takes several minutes.

In [ ]:
scaled_split_dir = ROOT / "outputs" / "scaled_split"
if scaled_split_dir.exists() and any(scaled_split_dir.glob("*.npz")):
    print("outputs/scaled_split/ already populated — skipping preprocessing.")
    print("Delete the directory and re-run this cell to force a rebuild.")
else:
    run("02_data_pipeline/scaled_split.py")

## Step 1 — Train stage 1: GRU-Causal+
Warm-starts the plant from `outputs/pipeline/Re__reults_of_gru_after_wight_/gru_plant.pt`.

Outputs → `outputs/pipeline/gru_causal_plus/gru_causal_plus/`
- `gru_plant.pt`, `gru_ctrl_*.pt`, `gru_loss_curves.png`, `results.json`

In [ ]:
run("03_model/train_gru_causal_plus.py")

## Step 2 — Train stage 2: Scenario-weighted
Warm-starts plant + controllers from stage 1 output.

Outputs → `outputs/pipeline/gru_scenario_weighted/`
- `gru_plant.pt`, `gru_ctrl_*.pt`, `gru_loss_curves.png`, `results.json`

In [ ]:
run("03_model/train_gru_scenario_weighted.py")

## Step 3 — Evaluate checkpoints (NRMSE)
Computes NRMSE per PV, per scenario, and overall for each checkpoint.
Results saved to `eval_results.json` inside each checkpoint directory.

In [ ]:
# Stage 1
run("04_evaluate/evaluate_model.py",
    "--ckpt", "outputs/pipeline/gru_causal_plus/gru_causal_plus")

In [ ]:
# Stage 2
run("04_evaluate/evaluate_model.py",
    "--ckpt", "outputs/pipeline/gru_scenario_weighted")

## Step 4 — Attack detection
Uses `gru_scenario_weighted` checkpoint. Produces ROC, PR curve, confusion matrix, residual plots.

Outputs → `report_plots/figures/s3/`

In [ ]:
run("05_detect/sec3_detection.py")

## Step 5 — Attack classification
Trains Baseline / TRTS / Mixed classifiers. Saves the TRTS Random Forest to `outputs/classifiers/`.

Outputs:
- `report_plots/figures/s3c_*.png`
- `outputs/classifiers/trts_rf_classifier.pkl`
- `outputs/classifiers/trts_rf_scaler.pkl`

In [ ]:
run("05_detect/sec3_classification.py")

## Summary — final eval results

In [ ]:
import json

checkpoints = {
    "Stage 1 (gru_causal_plus)":       ROOT / "outputs/pipeline/gru_causal_plus/gru_causal_plus/eval_results.json",
    "Stage 2 (gru_scenario_weighted)": ROOT / "outputs/pipeline/gru_scenario_weighted/eval_results.json",
}

for name, path in checkpoints.items():
    if path.exists():
        data = json.loads(path.read_text())
        mean_nrmse = data.get("mean_val_nrmse") or data.get("mean_nrmse") or "(see file)"
        print(f"{name}: mean NRMSE = {mean_nrmse}")
    else:
        print(f"{name}: eval_results.json not found yet")